# **From Here**

In [ ]:
%%capture
# 1) Remove any old versions
!pip uninstall -y unsloth unsloth_zoo bitsandbytes

# 2) Upgrade pip
!pip install --upgrade pip

# 3) Install bitsandbytes explicitly (for 4-bit)
!pip install "bitsandbytes>=0.43.3"

# 4) Install latest unsloth_zoo and unsloth from GitHub
!pip install --upgrade --no-cache-dir "git+https://github.com/unslothai/unsloth_zoo.git"
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 5) HF stack
!pip install -q transformers accelerate huggingface_hub


In [ ]:
# 2) IMPORTS
import unsloth
from unsloth import FastLanguageModel
import torch

In [ ]:
# 3) CONFIG
model_name = "tuirdas/deepseek-r1-medical-lora"  # or your other LoRA repo
max_seq_length = 2048
max_new_tokens = 512

print("🏥 Medical AI Inference")
print(f"Model: {model_name}")


In [ ]:
# 4) LOAD MODEL
print(f"Loading model: {model_name}...")
print("This may take 1–2 minutes on first run (model download).")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,          # auto
    load_in_4bit = True,   # fast & memory‑efficient
)

FastLanguageModel.for_inference(model)
# after FastLanguageModel.from_pretrained(...)
model_lora = model  # so the API code works

print("\n✅ Model loaded and ready!")
print(f"   Parameters: {model.num_parameters():,}")
print(f"   Max sequence: {max_seq_length} tokens")
print(f"   Device: {model.device}")

In [ ]:
# 5) CHAT FUNCTIONS
def medical_chat(question: str,
                 max_tokens: int = max_new_tokens,
                 temperature: float = 0.7) -> str:
    prompt = f"""<｜User｜>
{question}

<｜Assistant｜>
<think>
Medical Reasoning Process:
"""
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_tokens,
            temperature = temperature,
            do_sample = True,
            top_p = 0.9,
            top_k = 50,
            repetition_penalty = 1.1,
            pad_token_id = tokenizer.eos_token_id,
        )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "</think>" in full_response:
        parts = full_response.split("</think>")
        if len(parts) > 1:
            answer = parts[1].strip()
            answer = answer.replace("Expert Medical Answer:", "").strip()
            answer = answer.replace("<｜Assistant｜>", "").strip()
            return answer

    return full_response.replace(prompt, "").strip()

def medical_chat_with_reasoning(
    question: str,
    max_tokens: int = max_new_tokens,
    temperature: float = 0.7,
) -> dict:
    prompt = f"""<｜User｜>
{question}

<｜Assistant｜>
<think>
Medical Reasoning Process:
"""

    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_tokens,
            temperature = temperature,
            do_sample = True,
            top_p = 0.9,
            top_k = 50,
            repetition_penalty = 1.1,
            pad_token_id = tokenizer.eos_token_id,
        )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove the prompt from the front (if present)
    content = full_response.replace(prompt, "").strip()

    reasoning = ""
    answer = ""

    # Try to extract reasoning + final answer using <think> tags
    if "<think>" in content and "</think>" in content:
        before_think, after_think = content.split("<think>", 1)
        reasoning_block, after_reasoning = after_think.split("</think>", 1)
        reasoning = reasoning_block.replace("Medical Reasoning Process:", "").strip()
        answer = after_reasoning.replace("Expert Medical Answer:", "").strip()
    else:
        # Fallback: treat everything as answer if tags missing
        answer = content.strip()

    return {
        "reasoning": reasoning,
        "answer": answer,
        "full_response": full_response,
    }


In [ ]:
q1 = "I have headache and nausea for 3 days with no fever. What could this be?"
print(medical_chat(q1))

In [ ]:
q2 = "I am a 35‑year‑old male. For the last 3 months I feel very thirsty, pee many times a day including at night, and have lost weight even though I eat normally. I often feel tired and sometimes have blurry vision. I do not know of any long‑term diseases and I am not on regular medicine. What conditions should be considered and what kind of tests should I ask my doctor for?"
res = medical_chat_with_reasoning(q2)
print("Reasoning:\n", res["reasoning"])
print("\nAnswer:\n", res["answer"])

In [ ]:
# ─── 12. FastAPI Inference Server ───────────────────────────────────────────
# Runs directly on PORT 8000 — no ngrok needed.
# Every request and response is printed to the terminal with timing.

!pip install -q fastapi uvicorn nest_asyncio

import time
import logging
from datetime import datetime, timezone
from threading import Thread

import torch
import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel

nest_asyncio.apply()

# ── Terminal logger ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("medical-api")

# ── App ──────────────────────────────────────────────────────────────────────
app = FastAPI(title="Medical AI Inference API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Request / Response logging middleware ────────────────────────────────────
@app.middleware("http")
async def log_requests(request: Request, call_next):
    start = time.perf_counter()
    body_bytes = await request.body()
    body_preview = body_bytes.decode("utf-8", errors="replace")[:300] or "—"
    log.info(
        f"→ {request.method} {request.url.path}  "
        f"client={request.client.host}  body={body_preview}"
    )
    response = await call_next(request)
    ms = (time.perf_counter() - start) * 1000
    log.info(
        f"← {request.method} {request.url.path}  "
        f"status={response.status_code}  [{ms:.0f} ms]"
    )
    return response

# ── Schemas ──────────────────────────────────────────────────────────────────
class ChatRequest(BaseModel):
    message: str
    max_tokens: int = 512

class ChatResponse(BaseModel):
    reply: str
    model: str = "deepseek-r1-medical-lora"

# ── Endpoints ────────────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model": "loaded",
        "device": str(model.device),
    }

@app.post("/api/v1/chat/message", response_model=ChatResponse)
async def chat_message(request: ChatRequest):
    try:
        prompt = (
            f"<｜User｜>\n"
            f"{request.message}\n\n"
            "<｜Assistant｜>\n"
            "<think>\n"
            "Medical Reasoning Process:\n"
        )
        inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model_lora.generate(
                **inputs,
                max_new_tokens=request.max_tokens,
                temperature=0.7,
                top_p=0.9,
                top_k=50,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )

        full = tokenizer.decode(outputs[0], skip_special_tokens=True)
        content = full.replace(prompt, "").strip()
        answer = content.split("</think>", 1)[-1].strip() if "</think>" in content else content

        log.info(f"   reply_preview={answer[:120].replace(chr(10), ' ')}...")
        return ChatResponse(reply=answer)

    except Exception as exc:
        log.error(f"❌ inference error: {exc}")
        raise HTTPException(status_code=500, detail=str(exc))

# ── Start server ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)